# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [25]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('GROQ_API_KEY')

if api_key and api_key.startswith('gsk_') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'openai/gpt-oss-120b'
openai = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)


API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [6]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [7]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [8]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [26]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'proficient page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'avatar page', 'url': 'https://edwarddonner.com/avatar/'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [27]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [28]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling openai/gpt-oss-120b
Found 6 relevant links


{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'services page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [29]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling openai/gpt-oss-120b
Found 5 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [30]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [31]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling openai/gpt-oss-120b
Found 9 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
convaiinnovations/laya
Updated
2 minutes ago
•
3.01k
Qwen/Qwen-Image-2.1
Updated
3 days ago
•
28.4k
•
1.98k
prism-ml/Ternary-Bonsai-2-27B-gguf
Updated
6 days ago
•
2.82M
•
1.93k
XingChen-AGI/Xing4.0-29B-A4B
Updated
5 days ago
•
39k
•
1.5k
abenzerps/Qwen-Image-2.1-Uncensored-GG

In [39]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [33]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [34]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling openai/gpt-oss-120b
Found 6 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nconvaiinnovations/laya\nUpdated\n10 minutes ago\n•\n3.02k\nQwen/Qwen-Image-2.1\nUpdated\n3 days ago\n•\n28.4k\n•\n1.

In [35]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [36]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling openai/gpt-oss-120b
Found 8 relevant links


**Hugging Face – The AI Community Building the Future**  
*Your partner for open‑source, collaborative, and ethical AI*  

---  

### 🌟 Company Overview  
Hugging Face is the world’s leading collaboration platform for the machine‑learning community. From 2 M+ open‑source models and 500 k+ datasets to 1 M+ interactive AI applications (Spaces), the Hub is the central place where engineers, scientists, and creators **share, discover, experiment, and deploy** cutting‑edge AI.  

### 🎯 Mission & Vision  
- **Open & Ethical AI:** Empower a transparent, community‑driven AI ecosystem that prioritises fairness and responsibility.  
- **Accelerate Innovation:** Provide the tools, infrastructure, and community that let anyone move from idea to production faster.  
- **Democratise Machine Learning:** Lower the barrier to entry for the next generation of ML engineers and researchers.  

### 🛠️ Core Platform & Products  

| Pillar | What It Offers | Typical Users |
|--------|----------------|---------------|
| **Models** | 2 M+ pre‑trained models (text, vision, audio, RL, diffusion) searchable by task, language, and size. | Researchers, developers, enterprises building AI‑powered products. |
| **Datasets** | 500 k+ curated datasets with versioning, licensing, and community contributions. | Data scientists, academic labs, startups needing high‑quality training data. |
| **Spaces** | Hosted, interactive AI apps (e.g., image generation, video synthesis, decision tools) with zero‑setup deployment. | Product teams, demo creators, educators. |
| **Buckets** | Scalable storage for large model weights, fine‑tuned checkpoints, and data assets. | Companies requiring secure, high‑throughput storage for AI pipelines. |
| **Enterprise Solutions** | Hugging Face PRO, Enterprise Support, Inference Endpoints, Private Hub, and custom consulting. | Large organisations, regulated industries, AI‑first businesses. |
| **Community Tools** | Docs, Discord, Forum, GitHub, Blog, Daily Papers, and a vibrant open‑source contributor base. | Anyone wanting to learn, collaborate, or stay current with AI research. |

### 🤝 Community & Culture  

- **Collaboration‑First:** The Hub thrives on open contributions—anyone can host a model, dataset, or app with a single click.  
- **Transparency & Ethics:** A dedicated focus on responsible AI, open licensing, and community‑driven governance.  
- **Diverse Talent:** Engineers, research scientists, product designers, and ethicists work side‑by‑side in a flat, inclusive environment.  
- **Learning & Sharing:** Regular blog posts, tutorials, Discord chats, and a public forum keep the community informed and engaged.  

### 📊 Customers & Use Cases  

While Hugging Face’s primary audience is the open‑source community, its enterprise services power AI across many sectors:  

- **Tech & SaaS:** Deploying large language models for chatbots, code assistants, and content generation.  
- **Healthcare:** Secure model sharing for medical imaging and natural‑language clinical note analysis.  
- **Finance:** Risk‑assessment models and fraud‑detection pipelines built on curated datasets.  
- **Education:** Interactive Spaces used for hands‑on AI labs and research projects.  

### 👩‍💼 Careers & Opportunities  

Hugging Face is constantly expanding its team of **machine‑learning engineers, research scientists, product designers, community managers, and ethics specialists**.  

- **What We Look For:** Passion for open‑source, collaborative mind‑set, curiosity about cutting‑edge AI, and a commitment to ethical development.  
- **Where to Find Open Roles:** Visit the **Careers** section on the website for current openings, internship programs, and remote‑work opportunities.  
- **Why Join Us:** Work at the heart of the AI revolution, contribute to tools used by millions, and shape the future of responsible AI.  

### 📞 Get in Touch  

- **Website:** https://huggingface.co  
- **Community:** Discord, Forum, Twitter, LinkedIn  
- **Support:** Enterprise Support & Inference Providers for business‑critical workloads  
- **Brand Assets:** Official logos, colors, and brand guidelines are available for partners and media.  

---  

**Hugging Face**—where open collaboration meets world‑class AI. Join the community, build the future, and make AI accessible to everyone.  

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [37]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [38]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling openai/gpt-oss-120b
Found 7 relevant links


**Hugging Face – The AI Community Building the Future**  

---  

### Who We Are  
Hugging Face is the world‑wide collaboration platform for the machine‑learning ecosystem. Our hub lets anyone **share, discover, experiment, and build** open‑source models, datasets, and applications. With a fast‑growing community of researchers, engineers, and creators, we empower the next generation of AI talent to develop an **open, ethical, and responsible** AI future.  

### Core Culture  
- **Open Collaboration:** All models, datasets, and Spaces are publicly accessible, encouraging peer review and rapid iteration.  
- **Community‑First:** Over **2 million+ models**, **500 k+ datasets**, and **1 million+ AI applications** live on our platform, driven by contributions from a global community.  
- **Ethical AI:** We champion transparency, reproducibility, and responsible AI practices in every tool we release.  
- **Innovation at Scale:** Our science team pushes the frontier of ML research while providing production‑ready libraries (e.g., 🤗 Transformers, 🤗 Datasets).  

### What We Offer  

| Category | Highlights |
|----------|------------|
| **Models** | Browse >2 M pre‑trained models (e.g., Qwen‑Image‑2.1, Xing4.0‑29B) for vision, language, multimodal tasks. |
| **Datasets** | Access 500 k+ curated datasets, from arXiv papers to Wikipedia dumps. |
| **Spaces** | Deploy interactive AI apps instantly; 1 M+ community‑built demos (image generation, video synthesis, decision systems). |
| **Buckets & Storage** | Scalable data storage for large‑scale training pipelines. |
| **Enterprise Solutions** | Hugging Face PRO, Enterprise Support, Inference Endpoints, and custom Inference Providers for secure, production‑grade deployments. |
| **Developer Tools** | Comprehensive Docs, SDKs, GitHub integration, Discord & Forum for real‑time help. |
| **Learning Resources** | Daily Papers, Blog, Learn portal, and community‑run tutorials. |

### Who Uses Hugging Face  

- **Tech Companies & Start‑ups** – Build and host proprietary models behind secure inference endpoints.  
- **Enterprises** – Leverage Hugging Face PRO for compliance, scaling, and dedicated support.  
- **Researchers & Academics** – Publish papers, share reproducible datasets, and benchmark innovations.  
- **Developers & Creators** – Rapidly prototype AI‑driven apps via Spaces without managing infrastructure.  

*(Exact customer names are not listed publicly, but the platform powers AI workloads for dozens of Fortune‑500 firms, AI‑first startups, and leading research labs worldwide.)*  

### Careers & Growth Opportunities  

- **Join a Mission‑Driven Team:** Work alongside world‑class scientists, engineers, and product designers shaping the AI frontier.  
- **Roles Across the Stack:** Open positions in research, engineering, product, design, community, and enterprise sales.  
- **Inclusive Culture:** We value diversity, transparency, and a collaborative mindset—every voice helps steer the future of AI.  
- **Learn While You Contribute:** Access internal learning paths, mentorship, and the broader Hugging Face community to accelerate your career.  

> *“Hugging Face is the collaboration platform for the machine learning community… to build an open and ethical AI future together.”* – Hugging Face Bio  

### Get In Touch  

- **Website:** https://huggingface.co  
- **Community:** Discord, Forum, GitHub, Twitter, LinkedIn  
- **Support:** Enterprise Support plans and a public help center  
- **Brand Assets:** Official logos, color palettes, and brand guidelines available for download.  

---  

**Ready to collaborate, innovate, and shape the next era of AI?** Explore our models, datasets, and Spaces today, or reach out to discuss partnership and career opportunities.

In [40]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling openai/gpt-oss-120b
Found 8 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


## 🎉 Welcome to **Hugging Face** – Where AI Gets a Big, Friendly Squeeze!  

*The AI community building the future, one model, dataset, and meme‑filled Discord chat at a time.*

---

### 🤖 What We Do (In Plain English)

| **What** | **Why It’s Cool** | **Numbers That Make You Say “Whoa!”** |
|----------|-------------------|--------------------------------------|
| **Models** | 2 M+ open‑source ML models you can fork, fine‑tune, or simply stare at in awe. | 28 k ⭐ on the hottest Qwen‑Image‑2.1 model |
| **Datasets** | 500 k+ public datasets – from the entire Wikipedia dump to cutting‑edge research corpora. | 272 k ⭐ on the Wikipedia dataset |
| **Spaces** | “Serverless” playgrounds where anyone can spin up an AI app in a few clicks. | 1 M+ ready‑to‑run applications (think AI art generators, chatbots, video‑to‑audio converters…) |
| **Buckets** | Unlimited storage for your models & data, because “out of space” is a myth. | (Unlimited – we’re generous like that) |
| **Community** | Discord, forums, GitHub, daily papers, and a never‑ending stream of witty memes. | 106 k+ AI & ML enthusiasts following us |

> **TL;DR:** Think of Hugging Face as the bustling AI marketplace where researchers, startups, and the occasional cat‑video enthusiast all converge to share, remix, and launch the next big thing.

---

### 🎯 Who’s Already Hugging Us?

| **Customer Type** | **Why They’re Here** |
|-------------------|----------------------|
| **Start‑ups & Scale‑ups** | Need fast, reliable inference endpoints and a one‑stop shop for models & data. |
| **Enterprises** | Want secure, private “Buckets” and **Hugging Face PRO** for compliance and SLAs. |
| **Researchers & Academics** | Unlimited public repos, citation‑friendly datasets, and a vibrant open‑source community. |
| **Creative Coders** | Spaces let you spin up AI‑driven art, music, and video apps without a PhD. |
| **Investors** | A platform with >2 M models, 1 M+ apps, and a *growing* revenue runway powered by enterprise contracts. |

---

### 💰 For the Money‑Minded: Pricing & Enterprise

| **Plan** | **What You Get** |
|----------|------------------|
| **Free** | Unlimited public repos, community support, and a front‑row seat to the model zoo. |
| **Hugging Face PRO** | Private repos, priority inference, advanced analytics, and a secret handshake with the support team. |
| **Enterprise** | Custom SLAs, on‑prem buckets, dedicated account managers, and the ability to say “We built this in-house” (even though it lives on our cloud). |

*Ask us for a demo – we promise we won’t make you sign an NDA longer than a transformer’s attention span.*

---

### 👩‍💻 Join the Hugging Family (Careers)

| **Role** | **What You’ll Do** | **What We’re Looking For** |
|----------|-------------------|----------------------------|
| **ML Engineer** | Build, test, and ship models that power everything from chatbots to image generators. | Experience with PyTorch/TensorFlow, love for open‑source. |
| **Research Scientist** | Push the frontiers of NLP/vision/ multimodal AI and publish the next “state‑of‑the‑art”. | PhD or equivalent, curiosity, and a habit of tweeting your results. |
| **Community Manager** | Keep the Discord buzzing, organize hackathons, and turn strangers into Hugging Face fans. | Empathy, meme‑savvy, and the ability to speak fluent “GitHub”. |
| **Product Manager** | Translate community wishes into features, roadmaps, and shiny new UI widgets. | Business sense, data‑driven mindset, and a penchant for hugging (metaphorically). |
| **Design Engineer** | Craft UI/UX that makes exploring 2 M models feel like a walk in the park. | Portfolio that shows you can make complex data beautiful. |

*Perk highlights:* Unlimited coffee (virtual or real), flexible remote work, swag that actually fits on a hoodie, and the chance to say “I work at the company that invented the 🤗 emoji” at parties.

---

### 🌐 Get In Touch

- **Website:** https://huggingface.co  
- **Discord:** Join the #ai‑party (invite link on the site)  
- **Forum & GitHub:** Contribute, raise issues, or just brag about your latest model.  
- **Blog & Daily Papers:** Stay updated on the latest breakthroughs and occasional dad jokes.  

---

### 🎈 Bottom Line

> **Hugging Face** is the *social network* for AI—where models get to mingle, datasets go on dates, and developers leave with a warm, fuzzy feeling (and maybe a new open‑source project).  

Whether you’re a **customer** looking for scalable inference, an **investor** hunting the next AI unicorn, or a **talent** who wants to work where code is as collaborative as a group hug, we’ve got a spot reserved for you.  

*Come for the models, stay for the community, and leave with a smile that says, “I just hugged a transformer.”*  

---  

**Ready to join the hug?**  
👉 **Sign up** now or **talk to sales** and let’s make AI a little more human together.  

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>